In [ ]:
!pip install torch==2.4.1 torchvision==0.19.1 --quiet
!pip install diffusers==0.30.0 transformers==4.44.0 accelerate controlnet-aux xformers opencv-python --quiet

In [ ]:
from google.colab import auth
auth.authenticate_user()

import pandas as pd
from google.cloud import storage
import io, cv2, numpy as np
from PIL import Image

PROJECT = "skin-tone-classifier"
SRC_BUCKET = "fitz_clean"
DST_BUCKET = "fp17kc-synthetic"
IMAGE_PREFIX = "images/"
WATERMARK_CROP = 0.10
TEST_N = 5  # change to however many you want to eyeball

# Pull manifest directly from GCS (or upload test_manifest.csv to Colab)
# If you already have test_manifest.csv locally, just:
df = pd.read_csv("test_manifest.csv")
sample = df.sample(n=TEST_N, random_state=42).reset_index(drop=True)
print(sample[["md5hash", "fst_class", "disease_label", "assembled_prompt"]].to_string())

In [ ]:
import torch
from diffusers import StableDiffusionControlNetImg2ImgPipeline, ControlNetModel, UniPCMultistepScheduler
from controlnet_aux import HEDdetector

processor = HEDdetector.from_pretrained('lllyasviel/Annotators')
controlnet = ControlNetModel.from_pretrained(
    "lllyasviel/control_v11p_sd15_softedge",
    torch_dtype=torch.float16
)
pipe = StableDiffusionControlNetImg2ImgPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    controlnet=controlnet,
    torch_dtype=torch.float16
)
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
pipe.safety_checker = None
pipe.enable_model_cpu_offload()
print("Pipeline ready")

In [ ]:
import os
from IPython.display import display

gcs_client = storage.Client(project=PROJECT)

NEGATIVE = (
    "jewelry, rings, watch, bracelet, metal, silver, gold, shiny, blurry, "
    "low resolution, distorted, unnatural colors, deformed anatomy, extra digits, "
    "writing, overlay, text, logo, SFS, watermark"
)

# v9 config — matches your existing notebook
V9 = dict(
    version="v8",
    strength=0.25,
    guidance_scale=6.5,
    num_inference_steps=50,
    controlnet_conditioning_scale=0.6,
    prompt_suffix="neutral clinical background, clear lighting"
)

def download_pil(md5hash: str) -> Image.Image | None:
    try:
        blob = gcs_client.bucket(SRC_BUCKET).blob(f"{IMAGE_PREFIX}{md5hash}.jpg")
        data = blob.download_as_bytes()
        arr = np.frombuffer(data, dtype=np.uint8)
        bgr = cv2.imdecode(arr, cv2.IMREAD_COLOR)
        return Image.fromarray(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
    except Exception as e:
        print(f"  Download failed: {e}")
        return None

def preprocess(img: Image.Image) -> Image.Image:
    w, h = img.size
    img = img.crop((0, 0, w, int(h * (1 - WATERMARK_CROP))))
    w, h = img.size
    s = min(w, h)
    img = img.crop(((w-s)//2, (h-s)//2, (w+s)//2, (h+s)//2))
    return img.resize((512, 512))

def upload_pil(img: Image.Image, blob_path: str):
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=90)
    buf.seek(0)
    gcs_client.bucket(DST_BUCKET).blob(blob_path).upload_from_file(buf, content_type="image/jpeg")

os.makedirs("/content/outputs/v9", exist_ok=True)

for _, row in sample.iterrows():
    md5 = row["md5hash"]
    fst = row["fst_class"]
    prompt = f"{row['assembled_prompt']}, {V9['prompt_suffix']}"
    seed = int(row["seed"])

    print(f"\n→ {md5} | {fst} | seed={seed}")
    print(f"  {prompt[:100]}...")

    src = download_pil(md5)
    if src is None:
        continue

    img = preprocess(src)
    softedge = processor(img)
    generator = torch.Generator().manual_seed(seed)

    outputs = pipe(
        prompt=prompt,
        image=img,
        control_image=softedge,
        negative_prompt=NEGATIVE,
        strength=V9["strength"],
        guidance_scale=V9["guidance_scale"],
        num_inference_steps=V9["num_inference_steps"],
        controlnet_conditioning_scale=V9["controlnet_conditioning_scale"],
        num_images_per_prompt=1,   # bump to 4 once you're happy with results
        generator=generator,
    ).images

    out = outputs[0]
    local_path = f"/content/outputs/v9/{md5}_v0.jpg"
    out.save(local_path)

    # Upload to gs://fp17kc-synthetic/FP5/<md5>_v0.jpg
    blob_path = f"{fst}/{md5}_v0.jpg"
    #upload_pil(out, blob_path)
    print(f"  ✓ Saved → gs://{DST_BUCKET}/{blob_path}")
    display(src)
    display(out)